# ITCS 6162: Data Mining - Programming Assignment

**In this assignment, you will explore data analysis, recommendation algorithms, and graph-based techniques using the MovieLens dataset. Your tasks will range from basic data exploration to advanced recommendation models, including:**
- Data manipulation with pandas
- User-item collaborative filtering
- Similarity-based recommendation models
- A Pixie-inspired Graph-based recommendation using adjacency lists with weighted random walks (without using NetworkX)


#### **Dataset Files:**
- **`u.data`**: User-movie ratings (`user_id  movie_id  rating  timestamp`)
- **`u.item`**: Movie metadata (`movie_id | title | release date | IMDB_website`)
- **`u.user`**: User demographics (`user_id | age | gender | occupation | zip_code`)

## **Part 1: Exploring and Cleaning Data**

### Inspecting the Dataset Format

The dataset is not in a traditional CSV format. To examine its structure, use the following shell command to display the first 10 lines of the file:

```sh
!head <file_name>


**In the cells given below. Write the code to read the files.**

In [9]:
# u.data
import pandas as pd

with open("u.data", "r") as f:
    for _ in range(10):
        print(f.readline().strip())

196	242	3	881250949
186	302	3	891717742
22	377	1	878887116
244	51	2	880606923
166	346	1	886397596
298	474	4	884182806
115	265	2	881171488
253	465	5	891628467
305	451	3	886324817
6	86	3	883603013


In [10]:
# u.item

with open("u.item", "r", encoding="latin-1") as f:
    for _ in range(10):
        print(f.readline().strip())

1|Toy Story (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)|0|0|0|1|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0
2|GoldenEye (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?GoldenEye%20(1995)|0|1|1|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
3|Four Rooms (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|0|1|0|0
4|Get Shorty (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995)|0|1|0|0|0|1|0|0|1|0|0|0|0|0|0|0|0|0|0
5|Copycat (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Copycat%20(1995)|0|0|0|0|0|0|1|0|1|0|0|0|0|0|0|0|1|0|0
6|Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)|01-Jan-1995||http://us.imdb.com/Title?Yao+a+yao+yao+dao+waipo+qiao+(1995)|0|0|0|0|0|0|0|0|1|0|0|0|0|0|0|0|0|0|0
7|Twelve Monkeys (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Twelve%20Monkeys%20(1995)|0|0|0|0|0|0|0|0|1|0|0|0|0|0|0|1|0|0|0
8|Babe (1995)|01-Jan-1995||http://us.imdb.com/M/title-exact?Babe%20(1995)|0|0|0|0|1

In [11]:
# u.user

with open("u.user", "r") as f:
    for _ in range(10):
        print(f.readline().strip())

1|24|M|technician|85711
2|53|F|other|94043
3|23|M|writer|32067
4|24|M|technician|43537
5|33|F|other|15213
6|42|M|executive|98101
7|57|M|administrator|91344
8|36|M|administrator|05201
9|29|M|student|01002
10|53|M|lawyer|90703


#### Loading the Dataset with Pandas

Use **pandas** to load the dataset into a DataFrame for analysis. Follow these steps:  

1. Import the necessary library: `pandas`.  
2. Use `pd.read_csv()` (or an appropriate function) to read the dataset file.  
3. Ensure the dataset is loaded with the correct delimiter (e.g., `','`, `'\t'`,`'|'` , or another separator if needed).  
4. Select and display the first few rows using `.head()`.

Ensure that:  

- The `ratings` dataset is read from `"u.data"` using tab (`'\t'`) as a separator and column names (`"user_id"`, `"movie_id"`, `"rating"` and `"timestamp"`).  
- The `movies` dataset is read from `"u.item"` using `'|'` as a separator, use columns (`0`, `1`, `2`), encoding (`"latin-1"`) and name the columns (`movie_id`, `title`, and `release_date`).  
- The `users` dataset is read from `"u.user"` using `'|'` as a separator, use columns (`0`, `1`, `2`, `3`) and name the columns (`user_id`, `age`, `gender`, and `occupation`).

In [15]:
# ratings
ratings_df = pd.read_csv('u.data', sep='\t', names=["user_id", "movie_id", "rating", "timestamp"])
print(ratings_df.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [16]:
# movies
movies_df = pd.read_csv('u.item', sep='|', encoding='latin-1', usecols=[0, 1, 2], names=["movie_id", "title", "release_date"])
print(movies_df.head())

   movie_id              title release_date
0         1   Toy Story (1995)  01-Jan-1995
1         2   GoldenEye (1995)  01-Jan-1995
2         3  Four Rooms (1995)  01-Jan-1995
3         4  Get Shorty (1995)  01-Jan-1995
4         5     Copycat (1995)  01-Jan-1995


In [17]:
# users
users_df = pd.read_csv('u.user', sep='|', usecols=[0, 1, 2, 3], names=["user_id", "age", "gender", "occupation"])
print(users_df.head())

   user_id  age gender  occupation
0        1   24      M  technician
1        2   53      F       other
2        3   23      M      writer
3        4   24      M  technician
4        5   33      F       other


**Note:** As a **Bonus** task save the `ratings`, `movies` and `users` dataframe created into a `.csv` file format. <br>
**Hint:** Use the `to_csv()` function in pandas to save these DataFrames as CSV files.

In [19]:
# ratings
ratings_df.to_csv('ratings.csv', index=False)

In [20]:
# movies
movies_df.to_csv('movies.csv', index=False)

In [21]:
# users
users_df.to_csv('users.csv', index=False)

**Display the first 10 rows of each file.**

In [23]:
# ratings
print(ratings_df.head(10))

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
5      298       474       4  884182806
6      115       265       2  881171488
7      253       465       5  891628467
8      305       451       3  886324817
9        6        86       3  883603013


In [24]:
# movies
#print(movies_df.head(10))
print(movies_df[movies_df['title'] == "In the Company of Men (1997)"])


     movie_id                         title release_date
261       262  In the Company of Men (1997)  01-Aug-1997


In [25]:
# users
print(users_df.head(10))

   user_id  age gender     occupation
0        1   24      M     technician
1        2   53      F          other
2        3   23      M         writer
3        4   24      M     technician
4        5   33      F          other
5        6   42      M      executive
6        7   57      M  administrator
7        8   36      M  administrator
8        9   29      M        student
9       10   53      M         lawyer


### Data Cleaning and Exploration with Pandas  

After loading the dataset, it’s important to clean and explore the data to ensure consistency and accuracy. Below are key **pandas** functions for cleaning and understanding the dataset.

#### 1. Handle Missing Values  
- `df.dropna()` – Removes rows with missing values.  
- `df.fillna(value)` – Fills missing values with a specified value.  

#### 2. Remove Duplicates  
- `df.drop_duplicates()` – Drops duplicate rows from the dataset.  

#### 3. Handle Incorrect Data Types  
- `df.astype(dtype)` – Converts columns to the appropriate data type.  

#### 4. Filter Outliers (if applicable)  
- `df[df['column_name'] > threshold]` – Filters rows based on a condition.  

#### 5. Rename Columns (if needed)  
- `df.rename(columns={'old_name': 'new_name'})` – Renames columns for clarity.  

#### 6. Reset Index  
- `df.reset_index(drop=True, inplace=True)` – Resets the index after cleaning.  

### Data Exploration Functions  

To better understand the dataset, use these **pandas** functions:  

- `df.shape` – Returns the number of rows and columns in the dataset.  
- `df.nunique()` – Displays the number of unique values in each column.  
- `df['column_name'].unique()` – Returns unique values in a specific column.  

**Example Usage in Pandas:**  
```python
import pandas as pd

# Load dataset
df = pd.read_csv("your_file.csv")

# Drop missing values
df_cleaned = df.dropna()

# Remove duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# Convert 'timestamp' column to datetime format
df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'])

# Display dataset shape
print("Dataset shape:", df_cleaned.shape)

# Display number of unique values in each column
print("Unique values per column:\n", df_cleaned.nunique())

# Display unique movie IDs
print("Unique movie IDs:", df_cleaned['movie_id'].unique()[:10])  # Show first 10 unique movie IDs


**Note:** The functions mentioned above are some of the widely used **pandas** functions for data cleaning and exploration. However, it is not necessary that all of these functions will be required in the exercises below. Use them as needed based on the dataset and the specific tasks.

**Convert Timestamps into Readable dates.**

In [29]:
# ratings
ratings_df['timestamp'] = pd.to_datetime(ratings_df['timestamp'])
ratings_df['timestamp'] = ratings_df['timestamp'].dt.date



# Display first 10 rows to verify
print(ratings_df.head())


   user_id  movie_id  rating   timestamp
0      196       242       3  1970-01-01
1      186       302       3  1970-01-01
2       22       377       1  1970-01-01
3      244        51       2  1970-01-01
4      166       346       1  1970-01-01


**Check for Missing Values**

In [31]:
# ratings
print(ratings_df.isnull().sum())

user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64


In [32]:
# movies
print(movies_df.isnull().sum())



movie_id        0
title           0
release_date    1
dtype: int64


In [33]:
# users
print(users_df.isnull().sum())

user_id       0
age           0
gender        0
occupation    0
dtype: int64


**Print the total number of users, movies, and ratings.**

In [35]:
print(f"Total Users: {users_df['user_id'].nunique()}")

print(f"Total Movies: {movies_df['movie_id'].nunique()}")

print(f"Total Ratings: {ratings_df.shape[0]}")

Total Users: 943
Total Movies: 1682
Total Ratings: 100000


## **Part 2: Collaborative Filtering-Based Recommendation**

### **Create a User-Item Matrix**

#### Instructions for Creating a User-Movie Rating Matrix

In this exercise, you will create a user-movie rating matrix using **pandas**. This matrix will represent the ratings that users have given to different movies.

1. **Dataset Overview**:  
   The dataset has already been loaded. It includes the following key columns:
   - `user_id`: The ID of the user.
   - `movie_id`: The ID of the movie.
   - `ratings`: The rating the user gave to the movie.

2. **Create the User-Movie Rating Matrix**:  
   Use the **`pivot()`** function in **pandas** to reshape the data. Your goal is to create a matrix where:
   - Each **row** represents a **user**.
   - Each **column** represents a **movie**.
   - Each **cell** contains the **rating** that the user has given to the movie.

   Specify the following parameters for the `pivot()` function:
   - **`index`**: The `user_id` column (this will define the rows).
   - **`columns`**: The `movie_id` column (this will define the columns).
   - **`values`**: The `rating` column (this will fill the matrix with ratings).

3. **Inspect the Matrix**:  
   After creating the matrix, examine the first few rows of the resulting matrix to ensure it has been constructed correctly.

4. **Handle Missing Values**:  
   It's likely that some users have not rated every movie, resulting in `NaN` values in the matrix. You will need to handle these missing values. Consider the following options:
   - **Fill with 0**: If you wish to represent missing ratings as zeros (indicating no rating).
   - **Fill with the average rating**: Alternatively, replace missing values with the average rating for each movie.

**Create the user-movie rating matrix using the `pivot()` function.**

In [41]:
user_movie_matrix  = ratings_df.pivot(index='user_id', columns='movie_id', values='rating')
print(user_movie_matrix.head()) 
user_movie_matrix = user_movie_matrix.fillna(0)
#user_movie_matrix = user_movie_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)


movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
5          NaN   NaN   NaN   NaN  

**Display the matrix to verify the transformation.**

In [43]:
print(user_movie_matrix.head()) 

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   2.0  ...   
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5          4.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
5          0.0   0.0   0.0   0.0  

### **User-Based Collaborative Filtering Recommender System**

#### **Objective**
In this task, you will implement a **user-based collaborative filtering** movie recommendation system using the **Movie dataset**. The goal is to recommend movies to a user based on the preferences of similar users.

##### **Step 1: Import Required Libraries**
Before starting, ensure you have the necessary libraries installed. Use the following imports:

```python
import pandas as pd  # For handling data
import numpy as np   # For numerical computations
from sklearn.metrics.pairwise import cosine_similarity  # For computing user similarity
```

##### **Step 2: Compute User-User Similarity**
- We will use **cosine similarity** to measure how similar each pair of users is based on their movie ratings.
- Since `cosine_similarity` does not handle missing values (NaN), replace them with `0` before computation.

##### **Instructions:**
1. Fill missing values with `0` using `.fillna(0)`.
2. Compute similarity using `cosine_similarity()`.
3. Convert the result into a **Pandas DataFrame**, with users as both row and column labels.

##### **Hint:**  
You can achieve this using the following approach:

```python
user_similarity = cosine_similarity(user_movie_matrix.fillna(0))
user_sim_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)
```

##### **Step 3: Implement the Recommendation Function**
Now, implement the function `recommend_movies_for_user(user_id, num=5)` to recommend movies for a given user.

##### **Function Inputs:**
- `user_id`: The target user for whom we need recommendations.
- `num`: The number of movies to recommend (default is 5).

##### **Function Steps:**
1. Find **similar users**:
   - Retrieve the similarity scores for the given `user_id`.
   - Sort them in **descending** order (highest similarity first).
   - Exclude the user themselves.
   
2. Get the **movie ratings** from these similar users.

3. Compute the **average rating** for each movie based on these users' preferences.

4. Sort the movies in **descending order** based on the computed average ratings.

5. Retrieve the **top `num` recommended movies**.

6. Map **movie IDs** to their **titles** using the `movies` DataFrame.

7. Return the results as a **Pandas DataFrame** with rankings.

##### **Step 4: Return the Final Recommendation List**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

##### **Hint:** Your final DataFrame should be created like this:
```python
result_df = pd.DataFrame({
    'Ranking': range(1, num+1),
    'Movie Name': movie_names     
})
result_df.set_index('Ranking', inplace=True)
```

#### **Example: User-Based Collaborative Filtering**
```python
recommend_movies_for_user(10, num = 5)
```
**Output:**
```
| Ranking | Movie Name                     |
|---------|--------------------------------|
| 1       | In the Company of Men (1997)   |
| 2       | Misérables, Les (1995)         |
| 3       | Thin Blue Line, The (1988)     |
| 4       | Braindead (1992)               |
| 5       | Boys, Les (1997)               |


In [51]:
# Step 1: Import Required Libraries
import pandas as pd  # For handling data
import numpy as np   # For numerical computations
from sklearn.metrics.pairwise import cosine_similarity  # For computing user similarity

# Step 2: (Assuming user_movie_matrix and movies_df are already created and available)

# Fill missing ratings with 0 for similarity computation
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Compute cosine similarity between all users
user_similarity = cosine_similarity(user_movie_matrix_filled)
user_sim_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# Step 3: Implement the Recommendation Function
def recommend_movies_for_user(user_id, num=5):
    """
    Recommend movies to a user using user-based collaborative filtering.

    Parameters:
    - user_id (int): The target user's ID.
    - num (int): Number of movie recommendations to return.

    Returns:
    - pd.DataFrame: A DataFrame with 'Ranking' and 'Movie Name' columns.
    """

    # Find similar users:
    # Retrieve the similarity scores for the given user_id.
    similarity_scores = user_sim_df[user_id]

    # Sort them in descending order (highest similarity first).
    similarity_scores = similarity_scores.sort_values(ascending=False)

    # Exclude the user themselves.
    similarity_scores = similarity_scores.drop(user_id)

    # Get the movie ratings from these similar users.
    top_similar_users = similarity_scores.head(num)

    predicted_ratings = {}

    # Compute the average rating for each movie based on these users' preferences.
    for movie_id in user_movie_matrix.columns:
        # Only consider movies not rated by the target user
        if user_movie_matrix.loc[user_id, movie_id] == 0:
            weighted_sum = 0
            weight_total = 0

            for sim_user in top_similar_users.index:
                rating = user_movie_matrix.loc[sim_user, movie_id]
                if rating > 0:
                    sim_score = top_similar_users[sim_user]
                    weighted_sum += sim_score * rating
                    weight_total += abs(sim_score)

            if weight_total != 0:
                predicted_ratings[movie_id] = weighted_sum / weight_total

    # Sort the movies in descending order based on the computed average ratings.
    sorted_predictions = sorted(predicted_ratings.items(), key=lambda x: x[1], reverse=True)[:num]

    # Retrieve the top num recommended movies.
    if not sorted_predictions:
        print(f"No recommendations available for user {user_id}.")
        return pd.DataFrame(columns=['Movie Name'])

    movie_ids = [movie_id for movie_id, _ in sorted_predictions]

    # Map movie IDs to their titles using the movies DataFrame.
    movie_names = movies_df.set_index('movie_id').loc[movie_ids]['title'].values

    # Step 4: Return the Final Recommendation List
    # Return the results as a Pandas DataFrame with rankings.
    result_df = pd.DataFrame({
        'Ranking': range(1, len(movie_names) + 1),
        'Movie Name': movie_names
    })
    result_df.set_index('Ranking', inplace=True)

    return result_df

# Example usage:
recommendations = recommend_movies_for_user(10, num=5)
print(recommendations)


                           Movie Name
Ranking                              
1        In the Company of Men (1997)
2              Misérables, Les (1995)
3          Thin Blue Line, The (1988)
4                    Braindead (1992)
5                    Boys, Les (1997)


"""
###  User-Based Collaborative Filtering — Step-by-Step Explanation

---

### **Step 1: Import Required Libraries**

We begin by importing the essential libraries:

- `pandas` for data manipulation  
- `numpy` for numerical computations  
- `cosine_similarity` from `sklearn` to calculate user similarity based on their movie ratings  

---

### **Step 2: Prepare the User-Movie Matrix and Compute Similarity**

We fill missing values in the user-movie rating matrix with 0, treating unrated movies as no interaction. This allows us to safely compute cosine similarity between users.  
The similarity matrix is stored in a DataFrame for easy indexing by user IDs.

---

### **Step 3: Implement the Recommendation Function**

This function generates movie recommendations for a given user using user-based collaborative filtering. The logic inside follows the assignment’s structure:

- **3.1 Retrieve the similarity scores for the given `user_id`:**  
  We extract the similarity vector for the target user from the user similarity matrix.

- **3.2 Sort them in descending order (highest similarity first):**  
  This prioritizes the most relevant users based on behavioral similarity.

- **3.3 Exclude the user themselves:**  
  The user is removed from their own neighbor list to avoid self-comparison.

- **3.4 Get the movie ratings from these similar users:**  
  We select the top `num` most similar users and extract their movie ratings.

- **3.5 Compute the average rating for each movie based on these users' preferences:**  
  For each movie the target user has **not rated (i.e., rating is 0)**:
  
  - We look at the ratings of the top similar users for that movie.  
  - We **only consider ratings that are greater than 0**, meaning the user has actually rated the movie.  
  - We compute the **weighted average** of those ratings, using similarity scores as weights.  
  - This ensures that movies are only predicted if **at least one similar user has rated them**, and higher similarity gives more influence.

- **3.6 Sort the movies in descending order based on the computed average ratings:**  
  This ranks movies by how highly they are predicted to be rated by the target user.

- **3.7 Retrieve the top `num` recommended movies:**  
  We select the highest scoring movies from the prediction list.

- **3.8 Map movie IDs to their titles using the `movies_df` DataFrame:**  
  We convert the recommended movie IDs to their corresponding titles for final display.

---

### **Step 4: Return the Final Recommendation List**

We return the final result as a Pandas DataFrame with two columns:  
- **`Ranking`** — the rank of the recommendation  
- **`Movie Name`** — the title of the recommended movie  

This final output matches the format required by the assignment.
"""


### **Item-Based Collaborative Filtering Recommender System**

#### **Objective**
In this task, you will implement an **item-based collaborative filtering** recommendation system using the **Movie dataset**. The goal is to recommend movies similar to a given movie based on user rating patterns.

#### **Step 1: Import Required Libraries**
Although we have done this part already in the previous task but just to emphasize the importance reiterrating this part.

Before starting, ensure you have the necessary libraries installed. Use the following imports:

```python
import pandas as pd  # For handling data
import numpy as np   # For numerical computations
from sklearn.metrics.pairwise import cosine_similarity  # For computing item similarity
```

#### **Step 2: Compute Item-Item Similarity**
- We will use **cosine similarity** to measure how similar each pair of movies is based on their user ratings.
- Since `cosine_similarity` does not handle missing values (NaN), replace them with `0` before computation.
- Unlike user-based filtering, we need to **transpose** (`.T`) the `user_movie_matrix` because we want similarity between movies (columns) instead of users (rows).

##### **Instructions:**
1. Transpose the user-movie matrix using `.T` to make movies the rows.
2. Fill missing values with `0` using `.fillna(0)`.
3. Compute similarity using `cosine_similarity()`.
4. Convert the result into a **Pandas DataFrame**, with movies as both row and column labels.

##### **Hint:**  
You can achieve this using the following approach:

```python
item_similarity = cosine_similarity(user_movie_matrix.T.fillna(0))
item_sim_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)
```

#### **Step 3: Implement the Recommendation Function**
Now, implement the function `recommend_movies(movie_name, num=5)` to recommend movies similar to a given movie.

##### **Function Inputs:**
- `movie_name`: The target movie for which we need recommendations.
- `num`: The number of similar movies to recommend (default is 5).

##### **Function Steps:**
1. Find the **movie_id** corresponding to the given `movie_name` in the `movies` DataFrame.
2. If the movie is not found, return an appropriate message.
3. Extract the **similarity scores** for this movie from `item_sim_df`.
4. Sort the movies in **descending order** based on similarity (excluding the movie itself).
5. Retrieve the **top `num` similar movies**.
6. Map **movie IDs** to their **titles** using the `movies` DataFrame.
7. Return the results as a **Pandas DataFrame** with rankings.

#### **Step 4: Return the Final Recommendation List**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

##### **Hint:** Your final DataFrame should be created like this:
```python
result_df = pd.DataFrame({
    'ranking': range(1, num+1),
    'movie_name': movie_names
})
result_df.set_index('ranking', inplace=True)
```

#### **Example: Item-Based Collaborative Filtering**
```python
recommend_movies("Jurassic Park (1993)", num=5)
```
**Output:**
```
| Ranking | Movie Name                               |
|---------|------------------------------------------|
| 1       | Top Gun (1986)                           |
| 2       | Empire Strikes Back, The (1980)          |
| 3       | Raiders of the Lost Ark (1981)           |
| 4       | Indiana Jones and the Last Crusade (1989)|
| 5       | Speed (1994)                             |


In [60]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Step 2: Compute item-item similarity matrix
# Assumes user_movie_matrix and movies_df already exist
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Transpose so movies are rows (items), users are columns
movie_user_matrix = user_movie_matrix_filled.T

# Compute cosine similarity between items (movies)
item_similarity = cosine_similarity(movie_user_matrix)
item_sim_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)



###THIS CODE SNIPPET JUST BEFORE THE FUNCTION IS JUST FOR DEBUGGING PURPOSE###
# Get movie_id for "Jurassic Park (1993)"
jp_row = movies_df[movies_df['title'] == "Jurassic Park (1993)"]

if not jp_row.empty:
    jp_id = jp_row.iloc[0]['movie_id']
    
    # Get top 10 similar movie IDs and their similarity scores
    top_similar = item_sim_df.loc[jp_id].sort_values(ascending=False).drop(jp_id).head(10)
    
    # Print with movie titles
    print("\nTop 10 movies similar to 'Jurassic Park (1993)':\n")
    for movie_id, sim_score in top_similar.items():
        movie_title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
        print(f"{movie_id:<5} | {movie_title:<45} | Similarity: {sim_score:.6f}")
else:
    print("Movie not found.")
####                                                                         ####

# Step 3: Implement the Recommendation Function
def recommend_movies(movie_name, num=5):
    """
    Recommend movies similar to the given movie using item-based collaborative filtering.

    Parameters:
    - movie_name (str): The title of the movie to find recommendations for.
    - num (int): The number of similar movies to recommend.

    Returns:
    - pd.DataFrame: Top 'num' similar movies in ranked order.
    """

    # 3.1: Find the movie_id corresponding to the given movie_name
    match = movies_df[movies_df['title'] == movie_name]

    # 3.2: If the movie is not found, return a message
    if match.empty:
        print(f"Movie '{movie_name}' not found in the dataset.")
        return pd.DataFrame(columns=['Movie Name'])

    target_movie_id = match.iloc[0]['movie_id']

    # 3.3: Extract the similarity scores for this movie from item_sim_df
    if target_movie_id not in item_sim_df.index:
        print(f"No similarity data available for '{movie_name}'.")
        return pd.DataFrame(columns=['Movie Name'])

    similarity_scores = item_sim_df[target_movie_id]

    # 3.4: Sort the movies in descending order based on similarity (excluding itself)
    similarity_scores = similarity_scores.drop(target_movie_id)
    top_similar_ids = similarity_scores.sort_values(ascending=False).head(num).index

    # 3.5: Map movie IDs to their titles using the movies DataFrame (preserve order)
    similar_movie_names = [movies_df.set_index('movie_id').loc[mid]['title'] for mid in top_similar_ids]

    # Step 4: Return the Final Recommendation List
    result_df = pd.DataFrame({
        'ranking': range(1, len(similar_movie_names) + 1),
        'movie_name': similar_movie_names
    })
    result_df.set_index('ranking', inplace=True)

    return result_df

# Example 
recs = recommend_movies("Jurassic Park (1993)", num=5)
print(recs)




Top 10 movies similar to 'Jurassic Park (1993)':

161   | Top Gun (1986)                                | Similarity: 0.734101
568   | Speed (1994)                                  | Similarity: 0.721395
174   | Raiders of the Lost Ark (1981)                | Similarity: 0.715714
172   | Empire Strikes Back, The (1980)               | Similarity: 0.714439
210   | Indiana Jones and the Last Crusade (1989)     | Similarity: 0.704621
385   | True Lies (1994)                              | Similarity: 0.702316
204   | Back to the Future (1985)                     | Similarity: 0.693166
228   | Star Trek: The Wrath of Khan (1982)           | Similarity: 0.689716
403   | Batman (1989)                                 | Similarity: 0.687368
195   | Terminator, The (1984)                        | Similarity: 0.679616
                                        movie_name
ranking                                           
1                                   Top Gun (1986)
2                         

## **Part 3: Graph-Based Recommender (Pixie-Inspired Algorithm)**

### **Adjacency List**

#### **Objective**
In this task, you will preprocess the Movie dataset and construct a **graph representation** where:
- **Users** are connected to the movies they have rated.
- **Movies** are connected to users who have rated them.
  
This graph structure will help in exploring **user-movie relationships** for recommendations.

#### **Step 1: Merge Ratings with Movie Titles**
Since we have **movie IDs** in the ratings dataset but need human-readable movie titles, we will:
1. Merge the `ratings` DataFrame with the `movies` DataFrame using the `'movie_id'` column.
2. This allows each rating to be associated with a **movie title**.

#### **Hint:**
Use the following Pandas operation to merge:
```python
ratings = ratings.merge(movies, on='movie_id')
```


#### **Step 2: Aggregate Ratings**
Since multiple users may rate the same movie multiple times, we:
1. Group the dataset by `['user_id', 'movie_id', 'title']`.
2. Compute the **mean rating** for each movie by each user.
3. Reset the index to ensure we maintain a clean DataFrame structure.

#### **Hint:**  
Use `groupby()` and `mean()` as follows:
```python
ratings = ratings.groupby(['user_id', 'movie_id', 'title'])['rating'].mean().reset_index()
```

#### **Step 3: Normalize Ratings**
Since different users have different rating biases, we normalize ratings by:
1. **Computing each user's mean rating**.
2. **Subtracting the mean rating** from each individual rating.

#### **Instructions:**
- Use `groupby('user_id')` to group ratings by users.
- Apply `transform(lambda x: x - x.mean())` to adjust ratings.

#### **Hint:**  
Normalize ratings using:
```python
ratings['rating'] = ratings.groupby('user_id')['rating'].transform(lambda x: x - x.mean())
```
This ensures each user’s ratings are centered around zero, making similarity calculations fairer.

#### **Step 4: Construct the Graph Representation**
We represent the user-movie interactions as an **undirected graph** using an **adjacency list**:
- Each **user** is a node connected to movies they rated.
- Each **movie** is a node connected to users who rated it.

#### **Graph Construction Steps:**
1. Initialize an empty dictionary `graph = {}`.
2. Iterate through the **ratings dataset**.
3. For each `user_id` and `movie_id` pair:
   - Add the movie to the user’s set of connections.
   - Add the user to the movie’s set of connections.

#### **Hint:**  
The following code builds the graph:

```python
graph = {}
for _, row in ratings.iterrows():
    user, movie = row['user_id'], row['movie_id']
    if user not in graph:
        graph[user] = set()
    if movie not in graph:
        graph[movie] = set()
    graph[user].add(movie)
    graph[movie].add(user)
```

This results in a **bipartite graph**, where:
- **Users** are connected to multiple movies.
- **Movies** are connected to multiple users.

#### **Step 5: Understanding the Graph**
- **Nodes** in the graph represent **users and movies**.
- **Edges** exist between a user and a movie **if the user has rated the movie**.
- This structure allows us to find **users with similar movie tastes** and **movies frequently watched together**.

#### **Exploring the Graph**
- **Find a user’s rated movies:**  
  ```python
  user_id = 1
  print(graph[user_id])  # Movies rated by user 1
  ```

- **Find users who rated a movie:**  
  ```python
  movie_id = 50
  print(graph[movie_id])  # Users who rated movie 50
  ```

In [92]:
# Step 1: Merge Ratings with Movie Titles
# Merge ratings_df with movies_df using 'movie_id' to get human-readable movie titles
ratings = ratings_df.merge(movies_df, on='movie_id')

# Step 2: Aggregate Ratings
# Group by user, movie, and title to compute the mean rating in case of duplicates
ratings = ratings.groupby(['user_id', 'movie_id', 'title'])['rating'].mean().reset_index()

# Step 3: Normalize Ratings
# Normalize each user's ratings by subtracting their mean rating
ratings['rating'] = ratings.groupby('user_id')['rating'].transform(lambda x: x - x.mean())

# Step 4: Construct the Graph Representation (Adjacency List)
# Initialize an empty graph
graph = {}

# Create undirected bipartite edges between users and movies
for _, row in ratings.iterrows():
    user = row['user_id']
    movie = row['movie_id']
    
    # Add the movie to the user's connections
    if user not in graph:
        graph[user] = set()
    graph[user].add(movie)
    
    # Add the user to the movie's connections
    if movie not in graph:
        graph[movie] = set()
    graph[movie].add(user)

# Step 5: Explore the Graph
# Example 1: Movies rated by a user
user_id = 1
print(f"Movies rated by User {user_id}:", graph[user_id])

# Example 2: Users who rated a movie
movie_id = 50
print(f"Users who rated Movie {movie_id}:", graph[movie_id])


Movies rated by User 1: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217

###  Graph-Based Recommender — Steps 1 to 5 Explanation

---

### **Step 1: Merge Ratings with Movie Titles**

The ratings dataset contains only `movie_id`, which isn't human-readable.  
To make the data interpretable, I merged the `ratings_df` with `movies_df` on the `movie_id` column.  
This attaches each rating to its corresponding movie title.

---

### **Step 2: Aggregate Ratings**

Multiple users may rate the same movie more than once.  
To clean the data, I grouped it by `['user_id', 'movie_id', 'title']` and calculated the **average rating** given by each user to each movie.  
This helps reduce duplicate entries while preserving user preferences.

---

### **Step 3: Normalize Ratings**

Since users have different rating habits (some rate high, some rate low), I normalized the ratings by user.  
For each user, I computed their **mean rating**, and then subtracted it from each of their ratings.  
This makes the data more comparable across users by removing individual bias.

---

### **Step 4: Construct the Graph Representation**

I created an **undirected bipartite graph** using an adjacency list (Python dictionary).  
Each node in the graph is either a **user** or a **movie**.

For each user-movie pair in the dataset:
- I added an edge from the **user** to the **movie** they rated.
- I also added an edge from the **movie** back to the **user**.

This structure enables efficient traversal between users and the movies they interact with.

---

### **Step 5: Explore the Graph**

With the graph built, I can easily inspect relationships:

- `graph[user_id]` returns all **movies rated by that user**.
- `graph[movie_id]` returns all **users who rated that movie**.

This bipartite graph structure will serve as the foundation for random walk–based recommendation algorithms in the next steps.


### **Implement Weighted Random Walks**

#### **Random Walk-Based Movie Recommendation System (Weighted Pixie)**

#### **Objective**
In this task, you will implement a **random-walk-based recommendation algorithm** using the **Weighted Pixie** method. This technique uses a **user-movie bipartite graph** to recommend movies by simulating a random walk from a given user or movie.

#### **Step 1: Import Required Libraries**
Make sure you have the necessary libraries:

```python
import random  # For random walks
import pandas as pd  # For handling data
```

#### **Step 2: Implement the Random Walk Algorithm**
Your task is to **simulate a random walk** from a given starting point in the **bipartite user-movie graph**.

##### **Hints for Implementation**
- Start from **either a user or a movie**.
- At each step, **randomly move** to a connected node.
- Keep track of **how many times each movie is visited**.
- After completing the walk, **rank movies by visit count**.

#### **Step 3: Implement User-Based Recommendation**
**Hints:**
- Check if the `user_id` exists in the `graph`.
- Start a loop that runs for `walk_length` steps.
- Randomly pick a **connected node** (user or movie).
- Track how many times each **movie** is visited.
- Sort movies by visit frequency and return the **top N**.

#### **Step 4: Implement Movie-Based Recommendation**
**Hints:**
- Find the `movie_id` corresponding to the given `movie_name`.
- Ensure the movie exists in the `graph`.
- Start a random walk from that movie.
- Follow the same **tracking and ranking** process as the user-based version.

**Note:**  
**Your task:** Implement a function `weighted_pixie_recommend(user_id, walk_length=15, num=5)` or `weighted_pixie_recommend(movie_name, walk_length=15, num=5)`.  
**Implement either Step 3 or Step 4.**

#### **Step 5: Running Your Recommendation System**
Once your function is implemented, test it by calling:

##### **Example: User-Based Recommendation**
```python
weighted_pixie_recommend(1, walk_length=15, num=5)
```
| Ranking | Movie Name                     |
|---------|--------------------------------|
| 1       | My Own Private Idaho (1991)   |
| 2       | Aladdin (1992)                |
| 3       | 12 Angry Men (1957)           |
| 4       | Happy Gilmore (1996)          |
| 5       | Copycat (1995)                |


##### **Example: Movie-Based Recommendation**
```python
weighted_pixie_recommend("Jurassic Park (1993)", walk_length=10, num=5)
```
| Ranking | Movie Name                           |
|---------|-------------------------------------|
| 1       | Rear Window (1954)                 |
| 2       | Great Dictator, The (1940)         |
| 3       | Field of Dreams (1989)             |
| 4       | Casablanca (1942)                  |
| 5       | Nightmare Before Christmas, The (1993) |


#### **Step 6: Understanding the Results**
Your function should return a **DataFrame** structured as follows:

| Ranking | Movie Name |
|---------|-----------|
| 1       | Movie A   |
| 2       | Movie B   |
| 3       | Movie C   |
| 4       | Movie D   |
| 5       | Movie E   |

Each movie is ranked based on **how frequently it was visited** during the walk.

#### **Experiment with Different Parameters**
- Try different **`walk_length`** values and observe how it changes recommendations.
- Adjust the number of recommended movies (`num`).

In [127]:
# Step 1: Import Required Libraries
import random
import pandas as pd
from collections import defaultdict

# Step 2: Implement the Random Walk Algorithm
# This function simulates a smart-restart random walk on a user-movie bipartite graph.
def run_random_walk(start_node, walk_length, graph, restart_prob=0.3, is_user=True):
    visited_movies = defaultdict(int)
    current_node = start_node

    # 2.1: Define a smart restart pool:
    if is_user:
        rated = ratings_df[ratings_df['user_id'] == start_node]
        strong_movies = rated[rated['rating'] >= 4]['movie_id'].tolist()
        restart_candidates = strong_movies if strong_movies else [start_node]
    else:
        watchers = ratings_df[ratings_df['movie_id'] == start_node]
        frequent_users = watchers[watchers['rating'] >= 4]['user_id'].tolist()
        restart_candidates = frequent_users if frequent_users else [start_node]

    for _ in range(walk_length):
        # 2.2: Apply restart probability logic
        if random.random() < restart_prob:
            current_node = random.choice(restart_candidates)
            continue

        neighbors = list(graph.get(current_node, []))
        if not neighbors:
            break

        next_node = random.choice(neighbors)

        # 2.3: Count visits only if next node is a movie
        if next_node in movie_ids:
            visited_movies[next_node] += 1

        current_node = next_node

    return visited_movies

# Step 3 & 4: Implement Recommendation Function (User-Based or Movie-Based)
def weighted_pixie_recommend(start, walk_length=50, num=5, restart_prob=0.3):
    visited_movies = {}

    # Step 3: User-Based Recommendation
    if isinstance(start, int):
        # 3.1: Validate user existence in graph
        if start not in graph:
            return pd.DataFrame(columns=["Movie Name"])
        # 3.2: Perform random walk from user
        visited_movies = run_random_walk(start, walk_length, graph, restart_prob, is_user=True)

    # Step 4: Movie-Based Recommendation
    elif isinstance(start, str):
        # 4.1: Get movie ID from title
        match = movies_df[movies_df['title'] == start]
        if match.empty:
            return pd.DataFrame(columns=["Movie Name"])
        movie_id = match.iloc[0]['movie_id']

        # 4.2: Validate movie existence in graph
        if movie_id not in graph:
            return pd.DataFrame(columns=["Movie Name"])

        # 4.3: Perform random walk from movie
        visited_movies = run_random_walk(movie_id, walk_length, graph, restart_prob, is_user=False)

    else:
        return pd.DataFrame(columns=["Movie Name"])

    # Step 5: Running the Recommendation System
    # 5.1: Sort movies based on visit frequency
    sorted_movies = sorted(visited_movies.items(), key=lambda x: x[1], reverse=True)[:num]

    # Step 6: Understanding the Results
    # 6.1: Convert movie IDs to titles
    movie_names = [movies_df.set_index('movie_id').loc[mid]['title'] for mid, _ in sorted_movies]

    # 6.2: Create and return DataFrame with ranking
    result_df = pd.DataFrame({
        'Ranking': range(1, len(movie_names) + 1),
        'Movie Name': movie_names
    }).set_index('Ranking')

    return result_df

user_recommendations = weighted_pixie_recommend(1, walk_length=50, num=5)
movie_recommendations = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=50, num=5)

print("\n🎯 User-Based Recommendations for User ID = 1:")
print(user_recommendations)

print("\n🎬 Movie-Based Recommendations for 'Jurassic Park (1993)':")
print(movie_recommendations)

# Experiments with Weighted Pixie Parameters
# ---------------------------------------------

# Experiment 1: Default Settings
print("\n--- Experiment 1: User-Based | walk_length=50 | num=5 ---")
user_recs_1 = weighted_pixie_recommend(1, walk_length=50, num=5)
print(user_recs_1)

print("\n--- Experiment 1: Movie-Based | walk_length=50 | num=5 ---")
movie_recs_1 = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=50, num=5)
print(movie_recs_1)

# Experiment 2: Shorter Walk
print("\n--- Experiment 2: User-Based | walk_length=10 | num=5 ---")
user_recs_2 = weighted_pixie_recommend(1, walk_length=10, num=5)
print(user_recs_2)

print("\n--- Experiment 2: Movie-Based | walk_length=10 | num=5 ---")
movie_recs_2 = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=10, num=5)
print(movie_recs_2)

# Experiment 3: Deeper Walk
print("\n--- Experiment 3: User-Based | walk_length=100 | num=5 ---")
user_recs_3 = weighted_pixie_recommend(1, walk_length=100, num=5)
print(user_recs_3)

print("\n--- Experiment 3: Movie-Based | walk_length=100 | num=5 ---")
movie_recs_3 = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=100, num=5)
print(movie_recs_3)

# Experiment 4: More Recommendations
print("\n--- Experiment 4: User-Based | walk_length=50 | num=10 ---")
user_recs_4 = weighted_pixie_recommend(1, walk_length=50, num=10)
print(user_recs_4)

print("\n--- Experiment 4: Movie-Based | walk_length=50 | num=10 ---")
movie_recs_4 = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=50, num=10)
print(movie_recs_4)

# Experiment 5: Long Walk + More Results
print("\n--- Experiment 5: User-Based | walk_length=150 | num=10 ---")
user_recs_5 = weighted_pixie_recommend(1, walk_length=150, num=10)
print(user_recs_5)

print("\n--- Experiment 5: Movie-Based | walk_length=150 | num=10 ---")
movie_recs_5 = weighted_pixie_recommend("Jurassic Park (1993)", walk_length=150, num=10)
print(movie_recs_5)






🎯 User-Based Recommendations for User ID = 1:
                                  Movie Name
Ranking                                     
1                   Leaving Las Vegas (1995)
2                          Body Parts (1991)
3                      Reservoir Dogs (1992)
4                         Eve's Bayou (1997)
5        Highlander III: The Sorcerer (1994)

🎬 Movie-Based Recommendations for 'Jurassic Park (1993)':
                         Movie Name
Ranking                            
1           Schindler's List (1993)
2        Grosse Pointe Blank (1997)
3               Gang Related (1997)
4         Return of the Jedi (1983)
5                Money Train (1995)

--- Experiment 1: User-Based | walk_length=50 | num=5 ---
                          Movie Name
Ranking                             
1          African Queen, The (1951)
2                 Ulee's Gold (1997)
3        Doom Generation, The (1995)
4           Leaving Las Vegas (1995)
5                    Crooklyn (1994)

--- Expe

## Smart Pixie Recommender – Fully Detailed Step-by-Step Explanation

This markdown provides a complete explanation of the Smart Pixie-Inspired Random Walk Recommender, aligned with Steps 1–6 of the assignment, including all sub-steps.

---

### Step 1: Import Required Libraries

We import the necessary libraries to support graph-based recommendation:

- `random`: Enables randomness in the walk path.
- `pandas`: Handles user, movie, and ratings data using dataframes.
- `defaultdict` from `collections`: Tracks how frequently each movie is visited during the walk.

---

### Step 2: Implement the Random Walk Algorithm

The `run_random_walk()` function is responsible for performing a smart random walk across the user-movie graph.

Sub-Steps:

2.1: Start from either a user node or a movie node depending on the input.

2.2: At each walk step:
- With probability `restart_prob`, restart the walk to a contextually relevant node:
  - If starting from a user: restart to a movie rated ≥ 4 by that user.
  - If starting from a movie: restart to a user who rated that movie ≥ 4.
- If not restarting, randomly select a connected node (a neighbor) from the current node.

2.3: If the visited node is a movie, increment its visit count.

2.4: Continue this for a total of `walk_length` steps or until no further connections are available.

This produces a frequency map of visited movies based on their connectivity and contextual importance.

---

### Step 3: Implement User-Based Recommendation

The `weighted_pixie_recommend()` function handles both user- and movie-based recommendations. When a user ID is provided:

Sub-Steps:

3.1: Check if the user ID exists in the graph.

3.2: Start the smart random walk using the user as the initial node.

3.3: Use the output from the walk to count how often each movie was visited.

3.4: Sort the movies by visit count in descending order.

3.5: Select the top `num` movies to recommend.

---

### Step 4: Implement Movie-Based Recommendation

If the input is a movie title:

Sub-Steps:

4.1: Convert the movie title to its corresponding `movie_id`.

4.2: Check if the movie ID exists in the graph.

4.3: Perform a smart random walk starting from the movie node.

4.4: Collect frequency counts of visited movies.

4.5: Return the top `num` movies ranked by visit count.

---

### Step 5: Running Your Recommendation System

To run the recommendation system, you can test the function using:

```python
weighted_pixie_recommend(1, walk_length=50, num=5)                    # User-based example
weighted_pixie_recommend("Jurassic Park (1993)", walk_length=50, num=5)  # Movie-based example


## Results and Insights

---

### User-Based Collaborative Filtering

**Results and Insights:**

- The user-based collaborative filtering algorithm computes cosine similarity between users based on their rating vectors.
- Recommendations are made by aggregating weighted ratings from the top-k most similar users for each unrated movie.
- The model performs well when the user has rated a reasonable number of diverse movies, allowing similarity patterns to emerge.
- In sparse scenarios (users with few ratings), the similarity measure becomes less reliable, and recommendations may degrade in quality.
- The model effectively delivers personalized recommendations, but its dependency on user overlap makes it sensitive to data sparsity.

---

### Item-Based Collaborative Filtering

**Results and Insights:**

- This approach computes cosine similarity between movie rating vectors across users.
- Movies similar to a target movie are identified, and the highest-scoring ones are recommended.
- The model is independent of the current user and can be reused for multiple users without recomputation.
- Item-based filtering consistently recommends movies within the same genre or with similar popularity and reception.
- Its performance is robust in cases where target movies have been widely rated, but limited for obscure or niche films with fewer co-ratings.

---

### Graph-Based Recommender – Random Walk-Based Algorithm

**Results and Insights:**

- This method constructs a bipartite user-movie graph where nodes are connected based on rating interactions.
- The algorithm initiates a simple random walk from the user or movie node without any restart or prioritization.
- Movies are ranked by the number of times they are visited during the walk.
- It is computationally efficient and captures indirect associations through graph traversal.
- However, the lack of context-awareness results in some irrelevant or generic recommendations, especially for longer walks.
- The model occasionally visits unrelated parts of the graph, reducing recommendation precision.

---

### Graph-Based Recommender – Random Walk with Restart (Pixie-Inspired)

**Results and Insights:**

- This advanced graph-based method builds upon the simple walk by introducing a restart mechanism to guide the walk toward relevant nodes.
- For user-based recommendations, restarts occur to movies highly rated by the user. For movie-based recommendations, restarts target users who rated the movie highly.
- This strategy prevents the walk from drifting into irrelevant regions of the graph and increases the probability of visiting relevant movies.
- Recommendations are ranked by visit frequency, with higher-ranking movies being those most strongly connected to the user's or movie’s context.

**Experimental Observations:**

- Varying the `walk_length` and `num` parameters significantly impacts the recommendation output.
- Increasing `walk_length` allows the algorithm to explore a larger portion of the graph, resulting in more diverse recommendations. However, excessively long walks may reintroduce drift.
- Using a shorter `walk_length` (e.g., 15) yields tightly focused results that are highly relevant but potentially less diverse.
- Adjusting `num` (number of final recommendations) affects the breadth of suggestions but does not alter their relative quality, as rankings remain based on visitation frequency.
- Experimenting with different settings revealed that `walk_length=50` and `num=5` provided an optimal balance between relevance and diversity.
- The Pixie-inspired approach consistently outperformed the basic walk in aligning recommendations with user taste and thematic consistency.

---


## Explanation of Pixie-Inspired Algorithms

Pixie-inspired recommendation systems are graph-based algorithms designed to generate real-time, highly personalized recommendations using the structure of user-item 
interactions. Originally developed by Pinterest, these systems rely on representing users and items as nodes in a bipartite graph, where an edge represents a form of
interaction (such as a movie rating or a product view). The recommendation task is treated as a graph traversal problem, aiming to find items that are closely connected 
to a particular user or item node.

The core mechanism used in Pixie-inspired algorithms is the random walk. A random walk is a probabilistic process that starts at a given node and moves to one of its neighbors
at each step. In the context of recommendations, the algorithm may start from a user node or an item node and perform a number of steps, recording how many times each movie (or item) 
is visited during the walk. Items that are visited more frequently are assumed to be more relevant to the starting node and are returned as top recommendations.

To improve focus and relevance, a common enhancement is the use of random restarts. At each step, there is a probability that the walk will jump back to the starting node (or a similar node, such as a highly rated movie). 
This prevents the walk from drifting too far into unrelated areas of the graph and allows it to repeatedly explore the most relevant regions. Such restarts simulate the user's strong preferences and keep the recommendations 
closely tied to meaningful signals.

What makes Pixie-inspired algorithms particularly powerful is their ability to discover both direct and indirect relationships. They do not rely solely on similarity metrics or co-occurrence patterns, 
but can infer associations through paths in the graph, even if two items or users have no direct connection. This ability to uncover hidden structures makes Pixie suitable for sparse datasets and highly 
dynamic environments where explicit patterns may not be immediately visible.

Pixie-style algorithms have real-world applications in many major platforms. Pinterest uses them for pin recommendations, while companies like Netflix and Amazon have applied similar graph traversal methods 
for suggesting movies, shows, or products. These algorithms are valued for their scalability, real-time responsiveness, and flexibility in modeling complex user behavior, making them highly effective for
large-scale, production-grade recommendation systems.


---

## **Submission Requirements:**

To successfully complete this assignment, ensure that you submit the following:


### **1. Jupyter Notebook Submission**
- Submit a **fully completed Jupyter Notebook** that includes:
  - **All implemented recommendation functions** (user-based, item-based, and random walk-based recommendations).
  - **Code explanations** in markdown cells to describe each step.
  - **Results and insights** from running your recommendation models.


### **2. Explanation of Pixie-Inspired Algorithms (3-5 Paragraphs)**
- Write a **detailed explanation** of **Pixie-inspired random walk algorithms** used for recommendations.
- Your explanation should cover:
  - What **Pixie-inspired recommendation systems** are.
  - How **random walks** help in identifying relevant recommendations.
  - Any real-world applications of such algorithms in industry.


### **3. Report for the Submitted Notebook**
Your report should be structured as follows:

#### **Title: Movie Recommendation System Report**

#### **1. Introduction**
- Briefly introduce **movie recommendation systems** and why they are important.
- Explain the **different approaches used** (user-based, item-based, random-walk).

#### **2. Dataset Description**
- Describe the **MovieLens 100K dataset**:
  - Number of users, movies, and ratings.
  - What features were used.
  - Any preprocessing performed.

#### **3. Methodology**
- Explain the three recommendation techniques implemented:
  - **User-based collaborative filtering** (how user similarity was calculated).
  - **Item-based collaborative filtering** (how item similarity was determined).
  - **Random-walk-based Pixie algorithm** (why graph-based approaches are effective).
  
#### **4. Implementation Details**
- Discuss the steps taken to build the functions.
- Describe how the **adjacency list graph** was created.
- Explain how **random walks** were performed and how visited movies were ranked.

#### **5. Results and Evaluation**
- Present **example outputs** from each recommendation approach.
- Compare the different methods in terms of accuracy and usefulness.
- Discuss any **limitations** in the implementation.

#### **6. Conclusion**
- Summarize the key takeaways from the project.
- Discuss potential improvements (e.g., **hybrid models, additional features**).
- Suggest real-world applications of the methods used.

### **Submission Instructions**

- Submit `.zip` file consisting of Jupyter Notebook and all the datafiles (provided) and the ones saved [i.e. `users.csv`, `movies.csv` and `ratings.csv`]. Also, include the Report and Pixie Algorithm explanation document.
- [`Bonus 10 Points`] **Upload your Jupyter Notebook, Explanation Document, and Report** to your GitHub repository.
- Ensure the repository is public and contains:
  - `users.csv`, `movies.csv` and `ratings.csv` [These are the Dataframes which were created in part 1. Save and export them as a `.csv` file]
  - `Movie_Recommendation.ipynb`
  - `Pixie_Algorithm_Explanation.pdf` or `.md`
  - `Recommendation_Report.pdf` or `.md`
- **Submit the GitHub repository link in the cell below.**


#### **Example Submission Format**
```text
GitHub Repository: https://github.com/username/Movie-Recommendation
```

In [84]:
# Submit the Github Link here:


### **Grading Rubric: ITCS 6162 - Data Mining Assignment**


| **Category**                              | **Criteria**                                                     | **Points** |
|-------------------------------------------|----------------------------------------------------------------|------------|
| **Part 1: Exploring and Cleaning Data (15 pts)**  | Properly loads `u.user`, `u.movies`, and `u.item` datasets into DataFrames | 5 |
|                                           | Handles missing values, duplicates, and inconsistencies appropriately | 5 |
|                                           | Saves the cleaned datasets into CSV files: `users.csv`, `movies.csv`, `ratings.csv` | 5 |
| **Part 2: Collaborative Filtering-Based Recommendation (30 pts)** | Implements user-based collaborative filtering correctly | 10 |
|                                           | Implements item-based collaborative filtering correctly | 10 |
|                                           | Computes similarity measures accurately and provides valid recommendations | 10 |
| **Part 3: Graph-Based Recommender (Pixie-Inspired Algorithm) (35 pts)** | Constructs adjacency lists properly from user-movie interactions | 10 |
|                                           | Implements weighted random walk-based recommendation correctly | 15 |
|                                           | Explains and justifies the algorithm design choices (Pixie-inspired) | 10 |
| **Code Quality & Documentation (10 pts)** | Code is well-structured, efficient, and follows best practices | 5 |
|                                           | Markdown explanations and comments are clear and enhance understanding | 5 |
| **Results & Interpretation (5 pts)**      | Provides meaningful insights from the recommendation system's output | 5 |
| **Submission & Report (5 pts)**          | Submits all required files in the correct format (ZIP file with Jupyter notebook, processed CSV files, and project report) | 5 |
| **Total**                                 |                              | 100 |

#### **Bonus (10 pts)**
| **Category**                              | **Criteria**                                                     | **Points** |
|-------------------------------------------|----------------------------------------------------------------|------------|
| **GitHub Submission**                     | Provides a well-documented GitHub repository with CSV files, a structured README, and a properly formatted Jupyter Notebook | 10 |